In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler 

df = pd.read_csv("agriculture_eda_dataset.csv")

print(df.head())

          Region  Rainfall Temperature  Soil_Moisture    Soil_pH  Nitrogen  \
0         Punjab     650.0          24           45.0        7.2      80.0   
1         punjab     650.0         NaN           45.0  seven.two      80.0   
2        Haryana     600.0          25            NaN        7.5      75.0   
3  Uttar Pradesh     900.0         26C           50.0        6.8      70.0   
4      Rajasthan     -50.0          30           30.0        8.0      50.0   

   Phosphorus  Potassium  Irrigation  Crop_Area  ... Sunlight Humidity  \
0        40.0         45        95.0       4200  ...       8h     60.0   
1        40.0         45        95.0       4200  ...  8 hours     60.0   
2        38.0         42        90.0       3800  ...      8.2     58.0   
3        35.0         40        75.0       5000  ...      7.5     65.0   
4        25.0         30        40.0       3500  ...      NaN     40.0   

  Altitude  Wind_Speed  Groundwater Crop_Diversity  Mechanization  \
0      250       

In [2]:
word_to_num = {
    "seven.two" : 7.2,
    "eleven": 11.0,
    "one eighty": 180.0
}

def clean_and_convert(val):
    if pd.isna(val):
        return np.nan 
    val = str(val).lower().strip()

    if val in word_to_num:
        return word_to_num[val]

    val = val.replace('inr', '').replace('usd', '').replace('c', '')
    val = val.replace('h', '').replace('ours', '').replace('%', '')

    try:
        return float(val)
    except ValueError:
        return np.nan

# Apply cleaning to columns that had text/unit inconsistencies
cols_to_clean = ['Temperature', 'Soil_pH', 'Fertilizer', 'Sunlight', 'Crop_Diversity', 'Pesticide', 'Wind_Speed']
for col in cols_to_clean:
    df[col] = df[col].apply(clean_and_convert)

# Standardize the Region column text (lowercase and strip spaces)
df['Region'] = df['Region'].str.lower().str.strip()

# If Crop_Diversity was entered as percentages (e.g., 75 instead of 0.75), fix it
df.loc[df['Crop_Diversity'] > 1, 'Crop_Diversity'] = df.loc[df['Crop_Diversity'] > 1, 'Crop_Diversity'] / 100

In [3]:
# Fix logical errors (turn impossible values into NaNs so they can be imputed later)
df.loc[df['Rainfall'] < 0, 'Rainfall'] = np.nan
df.loc[df['Altitude'] < 0, 'Altitude'] = np.nan
df.loc[df['Temperature'] > 60, 'Temperature'] = np.nan
df.loc[df['Soil_pH'] > 14, 'Soil_pH'] = np.nan
df.loc[df['Fertilizer'] > 1000, 'Fertilizer'] = np.nan

# Drop the Date column as it usually isn't helpful for standard profile clustering
df_model = df.drop(columns=['Date'])

# Separate categorical and numerical data
regions = df_model['Region'] # Save regions for labeling clusters later
df_features = df_model.drop(columns=['Region'])

In [4]:
# Identify numeric columns
numeric_cols = df_features.select_dtypes(include=[np.number]).columns

# Impute missing values with the median
for col in numeric_cols:
    df_features[col] = df_features[col].fillna(df_features[col].median())

# Cap remaining extreme outliers at the 95th percentile to protect K-Means
for col in df_features.columns:
    upper_limit = df_features[col].quantile(0.95)
    df_features[col] = np.where(df_features[col] > upper_limit, upper_limit, df_features[col])

In [5]:
# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the features
scaled_data = scaler.fit_transform(df_features)

# Convert back to a DataFrame for easier handling
df_scaled = pd.DataFrame(scaled_data, columns=df_features.columns)

print("Preprocessing complete! Here is the scaled data:")
print(df_scaled.head())

Preprocessing complete! Here is the scaled data:
   Rainfall  Temperature  Soil_Moisture   Soil_pH  Nitrogen  Phosphorus  \
0 -1.089554    -1.416157      -0.475225  1.076735  1.096346    1.258313   
1 -1.089554    -0.209801      -0.475225  1.076735  1.096346    1.258313   
2 -1.244819    -0.812979       0.088186  1.681303  0.528419    0.781522   
3 -0.313230    -0.209801       0.088186  0.181078 -0.039508    0.066336   
4 -0.313230     1.599733      -2.165457  1.681303 -2.311217   -2.317617   

   Potassium  Irrigation  Crop_Area  Fertilizer  Pesticide  Sunlight  \
0   1.246708    1.909906   0.148902   -0.019123   1.593122  0.225443   
1   1.246708    1.909906   0.148902   -0.019123   1.593122  0.225443   
2   0.626053    1.579069  -0.307731    1.850150   0.881631  0.508786   
3   0.212282    0.476278   1.062169    0.255770   0.170139 -0.482916   
4  -1.856570   -2.096901  -0.650206   -1.943375  -1.252844 -0.057901   

   Humidity  Altitude  Wind_Speed  Groundwater  Crop_Diversity  Mec